# Phase 3 — CMB Angular Power Spectrum C^TT_ℓ

**Goal:** Convert the primordial power spectrum $P_\mathcal{R}(k)$ from Phase 2 into the CMB temperature angular power spectrum $C_\ell^{TT}$ using the Sachs-Wolfe approximation.

$$C_\ell^{TT} \approx \frac{2\pi^2}{9} \int \frac{dk}{k} \, P_\mathcal{R}(k) \, j_\ell^2(k\, r_{ls})$$

where $r_{ls} \approx 14\,000$ Mpc and $j_\ell$ is the spherical Bessel function.

**Input:** Golden trajectory from Phase 1 ($\phi_0 = 5.4$, $y_i = -0.05$, $\xi = 15000$).

**Output:** $C_\ell^{TT}$ for $\ell \in [2, 30]$ compared to $\Lambda$CDM baseline.

In [ ]:
import sys, os, glob, json
import numpy as np
import matplotlib.pyplot as plt


from scripts.camb_wrapper import compute_cl_sw, compute_cl_sw_baseline
from scripts.pspectrum_pipeline import load_pspectrum

plt.rcParams.update({'font.size': 14, 'axes.labelsize': 16, 'legend.fontsize': 12, 'figure.figsize': (12, 5)})

In [ ]:
pfile = sorted(glob.glob('../outputs/cmb_results/pspectra/*phi5.40_yi-0.050_run_*.json'))[-1]
data = load_pspectrum(pfile)
meta = data['metadata']

print(f"Trajectory: {meta['model']}, xi={meta['xi']}, phi0={meta['phi0']}, y0={meta['y0']}")
print(f"N_total = {meta['N_total']:.2f} e-folds, N_star = {meta['N_star']:.0f}")

In [ ]:
# Compton C_ell via SW integral
ELL_MAX = 30
ells_USR, C_ell_USR = compute_cl_sw(data, ell_max=ELL_MAX)
ells_LCDM, C_ell_LCDM = compute_cl_sw_baseline(data['k_phys'], ell_max=ELL_MAX)

# Conventir a D_ell = ell(ell+1) C_ell / (2pi) in muK^2
Tcmb = 2.7255
conv_factor = (Tcmb * 1e6)**2
D_ell_USR = ells_USR * (ells_USR + 1) / (2 * np.pi) * C_ell_USR * conv_factor
D_ell_LCDM = ells_LCDM * (ells_LCDM + 1) / (2 * np.pi) * C_ell_LCDM * conv_factor

print(f'C_ell ratio at low ell:')
for i in range(min(10, len(ells_USR))):
    r = C_ell_USR[i] / C_ell_LCDM[i]
    print(f'  ell={ells_USR[i]:2d}  USR/LCDM = {r:.4f}  ({((r-1)*100):+.1f}%)')

In [ ]:
# Fig 1: P_S(k) con caracteristica USR
k_phys = data['k_phys']
P_S = data['P_S']
As = meta['As_target']
k_piv = meta['k_pivot_phys']
ns_planck = 0.965

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: P_S(k) with USR
ax1.loglog(k_phys, P_S, 'b-', lw=2, label='USR (Higgs inf)')
ax1.loglog(k_phys, As * (k_phys/k_piv)**(ns_planck-1), 'k--', lw=1.5, label='LambdaCDM (ns=0.965)')
ax1.axvline(k_piv, color='gray', ls=':', alpha=0.5)
ax1.set_xlabel('k [Mpc^{-1}]')
ax1.set_ylabel('P_R(k)')
ax1.set_title('Primordial Power Spectrum')
ax1.legend()

# Right: local n_s(k)
logk = np.log(k_phys)
ns_local = 1 + np.diff(np.log(P_S)) / np.diff(logk)
k_mid = np.exp((logk[:-1] + logk[1:]) / 2)
ax2.semilogx(k_mid, ns_local, 'b-', lw=1.5)
ax2.axhline(ns_planck, color='k', ls='--', lw=1, label='ns(Planck)=0.965')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax2.set_xlabel('k [Mpc^{-1}]')
ax2.set_ylabel('n_s(k)')
ax2.set_title('Local Spectral Index')
ax2.legend()
ax2.set_ylim(0.94, 1.12)

plt.tight_layout()
plt.show()

In [ ]:
# Fig 2: C_ell TT from SW integral
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: D_ell absolute
ax1.semilogx(ells_USR, D_ell_USR, 'b-o', ms=5, label='USR (Higgs inf)')
ax1.semilogx(ells_LCDM, D_ell_LCDM, 'k--s', ms=4, label='LambdaCDM (ns=0.965)')
ax1.set_xlabel('Multipole ell')
ax1.set_ylabel('D_ell = ell(ell+1)C_ell/2pi [muK^2]')
ax1.set_title('CMB Temperature Power Spectrum')
ax1.legend()
ax1.set_xlim(2, 30)

# Right: D_ell ratio
ratio = D_ell_USR / D_ell_LCDM
ax2.semilogx(ells_USR, ratio, 'r-o', ms=6)
ax2.axhline(1.0, color='k', ls='--', lw=1)
ax2.fill_between(ells_USR, ratio, 1.0, alpha=0.3, color='red', where=(ratio < 1.0))
ax2.fill_between(ells_USR, ratio, 1.0, alpha=0.3, color='blue', where=(ratio > 1.0))
ax2.set_xlabel('Multipole ell')
ax2.set_ylabel('USR / LCDM')
ax2.set_title('Relative Suppression')
ax2.set_xlim(2, 30)
ax2.set_ylim(0.8, 1.05)

plt.tight_layout()
plt.show()

In [ ]:
# Save C_ell for Phase 4
from scripts.camb_wrapper import save_cl_results

output_path = save_cl_results(ells_USR, C_ell_USR, k_phys, P_S, meta,
    os.path.join('../outputs/cmb_results/c_ell', f'golden_phi{meta["phi0"]}_y0{meta["y0"]}.json'))
print(f'Saved C_ell to {output_path}')